1. Visualize detections (so you can figure out which class is “cell”)

In [1]:
import os
print("CWD =", os.getcwd())

CWD = /Users/reinakishida/Library/CloudStorage/Dropbox/Japan_Nation_Building/code


In [2]:
# Environment setup
import json
import os
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

In [3]:
# ======================
# EDIT THESE PATHS
# ======================
BASE_DIR = Path("/Users/reinakishida/Library/CloudStorage/Dropbox/Japan_Nation_Building") 
print("BASE_DIR =", BASE_DIR)

# roster_model_detection2.json: segmented using new model
JSON_PATH  = BASE_DIR / "data" / "dev" / "roster_model_detection2.json"

IMAGES_DIR = BASE_DIR / "output" / "purge_part2_1" / "pages_cropped" / "left"
OUT_DIR    = BASE_DIR / "output" / "roster_model_detection2_viz"
OUT_DIR.mkdir(parents=True, exist_ok=True)


MIN_SCORE = 0
ONLY_CLASS = None   # e.g., 3 to show only class 3, or None for all classes
# ======================

BASE_DIR = /Users/reinakishida/Library/CloudStorage/Dropbox/Japan_Nation_Building


In [4]:
def draw_boxes(img: Image.Image, dets, min_score=0.5, only_class=None) -> Image.Image:
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype("Arial.ttf", 18)
    except Exception:
        font = ImageFont.load_default()

    for d in dets:
        score = float(d.get("score", 0))
        cls = int(d.get("class_id", -1))
        if score < min_score:
            continue
        if only_class is not None and cls != only_class:
            continue

        x1, y1, x2, y2 = map(float, d["bbox"])
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1 + 2, y1 + 2), f"c{cls} {score:.2f}", fill="red", font=font)

    return img


def main():
    json_path = Path(JSON_PATH)
    images_dir = Path(IMAGES_DIR)
    out_dir = Path(OUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    if not json_path.exists():
        raise FileNotFoundError(f"JSON not found: {json_path}")
    if not images_dir.exists():
        raise FileNotFoundError(f"Images folder not found: {images_dir}")

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for img_name, dets in data.items():
        img_path = images_dir / img_name
        if not img_path.exists():
            print(f"[WARN] Missing image referenced by JSON: {img_path}")
            continue

        img = Image.open(img_path).convert("RGB")
        img = draw_boxes(img, dets, min_score=MIN_SCORE, only_class=ONLY_CLASS)

        out_path = out_dir / img_name
        img.save(out_path)
        print(f"[OK] wrote {out_path}")

    print("\nDone.")


if __name__ == "__main__":
    main()

[OK] wrote /Users/reinakishida/Library/CloudStorage/Dropbox/Japan_Nation_Building/output/roster_model_detection2_viz/spread_011_L.png
[OK] wrote /Users/reinakishida/Library/CloudStorage/Dropbox/Japan_Nation_Building/output/roster_model_detection2_viz/spread_013_L.png
[OK] wrote /Users/reinakishida/Library/CloudStorage/Dropbox/Japan_Nation_Building/output/roster_model_detection2_viz/spread_012_L.png

Done.
